# Manual Supercooling Annotation - Cell Workflow

This notebook intentionally does **not** use `ipywidgets`. It works by editing ordinary Python variables in cells and re-running them. That is less fancy than a widget dashboard, but it works in notebook frontends that show `VBox(...)` instead of real controls.

Workflow:

1. Run the setup cells.
2. Set `EXPERIMENT_ID`, `CHANNEL`, and optional zoom bounds.
3. Run the plot cell and inspect the trajectory.
4. Set `START_TIME_MIN`, `MAXIMUM_TIME_MIN`, and `END_TIME_MIN`.
5. Run the save cell.
6. Change channel/experiment and repeat.


In [ ]:
from pathlib import Path
import os
import json

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from freezeplots.paths import APP_DIR
from freezeplots.data_funcs import load_data_dict_hdf
from freezeplots.eval_funcs import get_tc_columns

# Inline images work in far more notebook frontends than widget dashboards.
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except NameError:
    pass

hdf5_path = Path(APP_DIR["data"]) / "raw_data.h5"
if not hdf5_path.exists():
    hdf5_path = Path("data/data_raw/raw_data.h5")
if not hdf5_path.exists():
    raise FileNotFoundError("Could not find data/data_raw/raw_data.h5")

data_dict = load_data_dict_hdf(str(hdf5_path))

with open("channel_layout.json", "r", encoding="utf-8") as file:
    channel_layout = json.load(file)

manual_event_columns = [
    "experiment_id",
    "channel",
    "start_time_min",
    "start_temp_c",
    "maximum_time_min",
    "maximum_temp_c",
    "end_time_min",
    "end_temp_c",
    "notes",
]
manual_events_path = Path(APP_DIR["output"]) / "manual_supercooling_events.csv"

if manual_events_path.exists():
    manual_events = pd.read_csv(manual_events_path)
else:
    manual_events = pd.DataFrame(columns=manual_event_columns)
manual_events = manual_events.reindex(columns=manual_event_columns)

print(f"Loaded {len(data_dict)} experiments from {hdf5_path}")
print(f"Manual annotations file: {manual_events_path}")


In [ ]:
def experiment_ids():
    return sorted(data_dict.keys())


def channels_for(experiment_id):
    return get_tc_columns(data_dict[experiment_id])


def channel_label(experiment_id, channel):
    return channel_layout.get(experiment_id, {}).get(channel, channel)


def nearest_time_temp(experiment_id, channel, time_min):
    df = data_dict[experiment_id]
    idx = (df["Time"] - float(time_min) * 60.0).abs().idxmin()
    return float(df.loc[idx, "Time"] / 60.0), float(df.loc[idx, channel])


def saved_row(experiment_id, channel):
    mask = (
        (manual_events["experiment_id"] == experiment_id)
        & (manual_events["channel"] == channel)
    )
    if mask.any():
        return manual_events.loc[mask].iloc[-1].to_dict()
    return None


def default_event_times(experiment_id, channel):
    df = data_dict[experiment_id]
    y = df[channel].to_numpy()
    t = (df["Time"] / 60.0).to_numpy()

    # Rough starting suggestion: largest positive jump, then local max afterward.
    dy = np.diff(y, prepend=y[0])
    start_idx = int(np.nanargmax(dy))
    search_end = min(len(y), start_idx + 250)
    if search_end <= start_idx:
        max_idx = start_idx
    else:
        max_idx = start_idx + int(np.nanargmax(y[start_idx:search_end]))

    end_idx = max_idx
    for idx in range(max_idx + 1, min(len(y), max_idx + 250)):
        if y[idx] < y[idx - 1]:
            end_idx = idx
            break

    return float(t[start_idx]), float(t[max_idx]), float(t[end_idx])


def show_overview():
    rows = []
    for exp in experiment_ids():
        rows.append({
            "experiment_id": exp,
            "channels": len(channels_for(exp)),
            "first_label": channel_label(exp, channels_for(exp)[0]),
        })
    return pd.DataFrame(rows)


def plot_channel(
    experiment_id,
    channel,
    start_time_min=None,
    maximum_time_min=None,
    end_time_min=None,
    zoom_start_min=None,
    zoom_end_min=None,
    notes="",
):
    df = data_dict[experiment_id]
    x = df["Time"] / 60.0
    y = df[channel]

    fig, ax = plt.subplots(figsize=(13, 5.5))
    ax.plot(x, y, color="#2f5597", linewidth=1.2, label=f"{channel} - {channel_label(experiment_id, channel)}")

    events = [
        ("start", start_time_min, "#1f77b4", "D", "Spike start / supercooling"),
        ("maximum", maximum_time_min, "#d62728", "o", "Spike maximum"),
        ("end", end_time_min, "#2ca02c", "s", "Spike end / decline"),
    ]
    for _name, event_time, color, marker, label in events:
        if event_time is None or pd.isna(event_time):
            continue
        snapped_time, temp = nearest_time_temp(experiment_id, channel, event_time)
        ax.axvline(snapped_time, color=color, linestyle="--", linewidth=1.0, alpha=0.8)
        ax.scatter([snapped_time], [temp], color=color, marker=marker, s=70, zorder=4, label=f"{label}: {snapped_time:.3f} min, {temp:.2f} C")

    if zoom_start_min is not None or zoom_end_min is not None:
        ax.set_xlim(
            zoom_start_min if zoom_start_min is not None else float(x.min()),
            zoom_end_min if zoom_end_min is not None else float(x.max()),
        )

    ax.set_title(f"{experiment_id} {channel}: {channel_label(experiment_id, channel)}")
    ax.set_xlabel("Time [min]")
    ax.set_ylabel("Temperature [deg C]")
    ax.grid(True, axis="both", alpha=0.25)
    ax.legend(loc="best", fontsize=9)
    if notes:
        ax.text(0.01, 0.02, notes, transform=ax.transAxes, fontsize=9, va="bottom")
    plt.tight_layout()
    plt.show()


def annotate_event(
    experiment_id,
    channel,
    start_time_min,
    maximum_time_min,
    end_time_min,
    notes="",
    export=True,
):
    global manual_events

    start_time_min, start_temp_c = nearest_time_temp(experiment_id, channel, start_time_min)
    maximum_time_min, maximum_temp_c = nearest_time_temp(experiment_id, channel, maximum_time_min)
    end_time_min, end_temp_c = nearest_time_temp(experiment_id, channel, end_time_min)

    row = {
        "experiment_id": experiment_id,
        "channel": channel,
        "start_time_min": start_time_min,
        "start_temp_c": start_temp_c,
        "maximum_time_min": maximum_time_min,
        "maximum_temp_c": maximum_temp_c,
        "end_time_min": end_time_min,
        "end_temp_c": end_temp_c,
        "notes": notes,
    }

    mask = (
        (manual_events["experiment_id"] == experiment_id)
        & (manual_events["channel"] == channel)
    )
    if mask.any():
        for col in manual_event_columns:
            manual_events.loc[mask, col] = row[col]
    else:
        manual_events = pd.concat([manual_events, pd.DataFrame([row])], ignore_index=True)

    if export:
        manual_events_path.parent.mkdir(parents=True, exist_ok=True)
        manual_events.to_csv(manual_events_path, index=False)

    plot_channel(
        experiment_id,
        channel,
        start_time_min=start_time_min,
        maximum_time_min=maximum_time_min,
        end_time_min=end_time_min,
        notes=notes,
    )
    return pd.DataFrame([row])


def next_channel(experiment_id, channel, offset=1):
    channels = channels_for(experiment_id)
    return channels[(channels.index(channel) + offset) % len(channels)]


## 1. Pick Experiment And Channel

Edit this cell, then run it. The output gives you suggested event times based on the largest positive temperature jump. These are only starting points for manual correction.


In [ ]:
# Edit these two values while annotating.
EXPERIMENT_ID = experiment_ids()[0]
CHANNEL = channels_for(EXPERIMENT_ID)[0]

print("Experiment:", EXPERIMENT_ID)
print("Channel:", CHANNEL, "-", channel_label(EXPERIMENT_ID, CHANNEL))
print("Available channels:", channels_for(EXPERIMENT_ID))
print("Saved row:", saved_row(EXPERIMENT_ID, CHANNEL))

START_TIME_MIN, MAXIMUM_TIME_MIN, END_TIME_MIN = default_event_times(EXPERIMENT_ID, CHANNEL)
print("Suggested times [min]:")
print("  START_TIME_MIN   =", START_TIME_MIN)
print("  MAXIMUM_TIME_MIN =", MAXIMUM_TIME_MIN)
print("  END_TIME_MIN     =", END_TIME_MIN)


## 2. Plot And Inspect

Adjust `ZOOM_START_MIN` and `ZOOM_END_MIN` if needed. Leave them as `None` for the full trajectory.


In [ ]:
ZOOM_START_MIN = None
ZOOM_END_MIN = None

plot_channel(
    EXPERIMENT_ID,
    CHANNEL,
    start_time_min=START_TIME_MIN,
    maximum_time_min=MAXIMUM_TIME_MIN,
    end_time_min=END_TIME_MIN,
    zoom_start_min=ZOOM_START_MIN,
    zoom_end_min=ZOOM_END_MIN,
)


## 3. Manually Correct Times And Save

Edit the three times below after inspecting the plot. Temperatures are snapped automatically from the nearest measured sample. Running this cell updates `output/manual_supercooling_events.csv`.


In [ ]:
# Edit these values manually, then run the cell.
START_TIME_MIN = START_TIME_MIN
MAXIMUM_TIME_MIN = MAXIMUM_TIME_MIN
END_TIME_MIN = END_TIME_MIN
NOTES = ""

annotate_event(
    EXPERIMENT_ID,
    CHANNEL,
    start_time_min=START_TIME_MIN,
    maximum_time_min=MAXIMUM_TIME_MIN,
    end_time_min=END_TIME_MIN,
    notes=NOTES,
    export=True,
)


## 4. Review Saved Annotations


In [ ]:
manual_events.sort_values(["experiment_id", "channel"]).reset_index(drop=True)


## Helpers

Use these snippets to move through the data.


In [ ]:
# Move to the next thermocouple channel, then re-run sections 1-3.
# CHANNEL = next_channel(EXPERIMENT_ID, CHANNEL)

# Show a compact overview of experiments.
# show_overview()

# Jump to a specific experiment/channel.
# EXPERIMENT_ID = "EJ01A1"
# CHANNEL = "TC1"
